In [ ]:
import pandas as pd
import numpy as np


In [ ]:
df=pd.read_csv('/content/loan_approval_dataset.csv')


In [ ]:
df.head()

,loan_id,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
0,1,2,Graduate,No,9600000,29900000,12,778,2400000,17600000,22700000,8000000,Approved
1,2,0,Not Graduate,Yes,4100000,12200000,8,417,2700000,2200000,8800000,3300000,Rejected
2,3,3,Graduate,No,9100000,29700000,20,506,7100000,4500000,33300000,12800000,Rejected
3,4,3,Graduate,No,8200000,30700000,8,467,18200000,3300000,23300000,7900000,Rejected
4,5,5,Not Graduate,Yes,9800000,24200000,20,382,12400000,8200000,29400000,5000000,Rejected


In [ ]:
#!pip install ydata_profiling
#from ydata_profiling import ProfileReport
#prof=ProfileReport(df)
#prof.to_file(output_file='output.html')

In [ ]:
df.head().shape


(5, 13)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest,chi2
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder


In [ ]:
X_train,X_test,y_train,y_test=train_test_split(df.iloc[:,0:12],df.iloc[:,-1],test_size=0.2)

In [ ]:
y_test.head()

,loan_status
3284,Approved
916,Approved
2415,Rejected
3015,Rejected
3077,Approved


In [ ]:
#one hot encoder
trf1=ColumnTransformer(
    [
        ('ohe_education',OneHotEncoder(sparse_output=False,handle_unknown='ignore'),[2]),
        ('ohe_self_employed',OneHotEncoder(sparse_output=False,handle_unknown='ignore'),[3])
        ],
        remainder='passthrough'
)


In [ ]:
#scaling
trf2=ColumnTransformer([
    ('scale',MinMaxScaler(),[0,1,4,5,6,7,8,9,10,11])
],remainder='passthrough')

In [ ]:
#feature selection
trf3=SelectKBest(score_func=chi2,k=8)

In [ ]:
#model train
#trf4=DecisionTreeClassifier()
#trf4=LogisticRegression()
trf4=RandomForestClassifier()

In [ ]:
pipe=Pipeline([
    ('trf1',trf1),
    ('trf2',trf2),
    ('trf3',trf3),
    ('trf4',trf4)])

In [ ]:
le=LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)
pipe.fit(X_train,y_train_encoded)

Pipeline(steps=[('trf1',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ohe_education',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  [2]),
                                                 ('ohe_self_employed',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  [3])])),
                ('trf2',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('scale', MinMaxScaler(),
                                                  [0, 1, 4, 5, 6, 7, 8, 9, 10,
                                                   11])])),
                ('trf3',
                 SelectKBest(k=8,
                             score_func=<function chi2 at 0x7f9bd26105e0>)),
                ('trf4', RandomForestClassifier())])

In [ ]:
y_pred=pipe.predict(X_test)


In [ ]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test_encoded,y_pred)*100

95.78454332552693

In [ ]:
import pickle
pickle.dump(pipe,open('pipe1.pkl','wb'))

In [ ]:
print(X_train.columns)

Index(['loan_id', ' no_of_dependents', ' education', ' self_employed',
       ' income_annum', ' loan_amount', ' loan_term', ' cibil_score',
       ' residential_assets_value', ' commercial_assets_value',
       ' luxury_assets_value', ' bank_asset_value'],
      dtype='object')
